# Plan 005.3: saving and inspecting a sync work list

This tutorial shows how Archiver keeps track of a proposed sync. Think of the
workspace as a saved work list: which files are already present, which still need
attention, and where an attempt stopped.

You only need basic Python and the project environment. Run the cells in order,
or restart the kernel and run them all again.

We will:

1. Make a tiny practice collection.
2. Create and inspect its work list.
3. Close and reopen the saved list.
4. Record a pretend problem and recover the unfinished attempt.
5. Discard the attempt and clean up.

Everything here lives in a temporary practice folder. Actual file copying belongs
to the next subplan, 005.4. This notebook demonstrates the new record-keeping tools.

## 1. Set up a practice folder

We make separate source and target folders, plus two catalogs. A cleanup helper
keeps track of everything we open, so the final cell can remove the whole practice
folder. These are new files created just for this lesson.

In [ ]:
from contextlib import ExitStack, closing
from pathlib import Path, PurePosixPath
from tempfile import TemporaryDirectory

import pandas as pd
from IPython.display import display

from archiver import (
    Catalog, ManagedRootAuthority, ManifestEntry,
    OperationError, OperationSpec, SyncWorkspace,
)

resources = ExitStack()
sandbox = Path(resources.enter_context(TemporaryDirectory(prefix="archiver-005-3-")))
source_root, target_root = sandbox / "source", sandbox / "target"
source_root.mkdir()
target_root.mkdir()
source = resources.enter_context(Catalog.create(sandbox / "source.sqlite"))
target = resources.enter_context(Catalog.create(sandbox / "target.sqlite"))
print("The practice folders and catalogs are ready.")

## 2. Add a few familiar files

Our source contains a note, a second copy of that note, an empty file, and a small
new file. The target already has the note under another name.

| Source file | Contents | Target already has these contents? |
| --- | --- | --- |
| `note.txt` | `hello` | Yes, in `saved-note.txt` |
| `second-note.txt` | `hello` | Yes, the same contents again |
| `empty.txt` | Empty | No |
| `nested/new.txt` | `new` | No |

We scan these practice folders once to give each catalog its starting inventory.

In [ ]:
source_files = {
    "note.txt": b"hello",
    "second-note.txt": b"hello",
    "empty.txt": b"",
    "nested/new.txt": b"new",
}
for name, data in source_files.items():
    file = source_root / name
    file.parent.mkdir(parents=True, exist_ok=True)
    file.write_bytes(data)
(target_root / "saved-note.txt").write_bytes(b"hello")
source.scan_directory(source_root)
target.scan_directory(target_root)

# These small checks let us confirm later that the lesson changed no data files.
def file_facts(root):
    return {
        file.relative_to(root).as_posix(): (
            file.read_bytes(), file.stat().st_mtime_ns, file.stat().st_mode
        )
        for file in root.rglob("*") if file.is_file()
    }

source_before, target_before = file_facts(source_root), file_facts(target_root)
source_database_before = source.database_path.read_bytes()
print("Source: 4 files. Target: 1 file.")

## 3. Name the source and target

An operation means one particular attempt. Its description records which catalog
and folder are the source, and which are the target. It also remembers their
starting inventories, so later work can notice if those inventories have changed.

Giving an attempt a name does not start it or grant permission to change files.

In [ ]:
source_scope = source.operation_scope(source_root)
target_scope = target.operation_scope(target_root)
spec = OperationSpec.sync(source_scope, target_scope)
print("Direction: source -> target")
print("This attempt has its own ID:", spec.operation_id)

## 4. Create the saved work list

The workspace is a temporary database beside our practice folders. It can survive
closing the program when an unfinished attempt needs attention.

We expect **three different sets of contents**, totalling **eight bytes**:
`hello` (5), `new` (3), and the empty file (0). The second note does not add another
set of contents. These totals include the note already at the target; the two
missing items together need only three bytes.

In [ ]:
workspace = resources.enter_context(SyncWorkspace.create(
    sandbox, spec, expected_count=3, expected_bytes=8,
))
workspace_directory = workspace.directory
print("Workspace state:", workspace.state)
assert workspace.state == "building"
assert workspace_directory.is_dir()

## 5. Put three items on the list

For this tiny example, we choose the items and destination names ourselves. This
is a teaching example, not the automatic sync planner planned for 005.4.

The note is marked **already present**. The empty file and the new file are
**waiting**. Neither has been copied. We use the catalog to identify each file's
contents, so there are no long hashes to type by hand.

In [ ]:
items = []
for name, already_present in (
    ("note.txt", True), ("empty.txt", False), ("nested/new.txt", False),
):
    file = source_root / name
    items.append(ManifestEntry(
        content_id=source.content_for_path(source_root, PurePosixPath(name)),
        size_bytes=file.stat().st_size,
        source_relative_path=PurePosixPath(name),
        source_mtime_ns=file.stat().st_mtime_ns,
        destination_relative_path=None if already_present else PurePosixPath(name),
        action="skip_present" if already_present else "copy",
        state="skipped" if already_present else "pending",
    ))
workspace.add(items[:2], batch_size=2)
workspace.add(items[2:], batch_size=2)
print("Saved three items, in two small batches.")

## 6. Read the whole-list totals

The totals describe the whole list, even when we look at only a small page later.

| Label below | Meaning |
| --- | --- |
| Different contents | Items on this work list, including already-present ones |
| Already present | Items the target catalog already knows about |
| Still waiting | Items whose work has not finished |
| Placed at target | Items recorded as successfully put in place |
| Added to inventory | Items confirmed as added to the target catalog |

Expect three items: one already present and two waiting. The last two counts
should both be zero.

In [ ]:
def show_counts(workspace):
    counts = workspace.summary()
    display(pd.DataFrame([
        ("Different contents", counts.planned),
        ("Already present", counts.skipped),
        ("Still waiting", counts.unattempted),
        ("Failed", counts.failed),
        ("Placed at target", counts.placed),
        ("Added to inventory", counts.registered),
    ], columns=["What we are counting", "Count"]))

show_counts(workspace)
counts = workspace.summary()
assert (counts.planned, counts.skipped, counts.unattempted) == (3, 1, 2)
assert counts.placed == counts.registered == 0

## 7. Read a little at a time

Here we read at most two items at once. A much longer list can be read in the same
way without asking Python to hold it all at once.

`pending` means waiting; `skipped` here means the contents are already cataloged
at the target. That catalog record alone does not prove the physical file is still
there.

In [ ]:
with closing(workspace.batches(batch_size=2)) as batches:
    for page_number, batch in enumerate(batches, start=1):
        print(f"Page {page_number}")
        display(pd.DataFrame([
            {"Source file": str(item.source_relative_path),
             "Target file": str(item.destination_relative_path) if item.destination_relative_path else "Already present",
             "Status": item.state}
            for item in batch
        ]))
        assert len(batch) <= 2
assert workspace.summary().planned == 3

## 8. Mark the list ready, then close it

Now we confirm that this small list is complete and all proposed destination
names are filled in. `ready` means the list is ready to use; it does not mean that
any files have been copied.

**Try predicting the result:** after we close the workspace, will its saved folder
still exist? Run the next cell to check.

In [ ]:
workspace.mark_ready(comparison_exhausted=True)
assert workspace.state == "ready"
workspace.close()
assert workspace_directory.exists()
print("Closed the workspace. Its saved list is still there.")

## 9. Reopen it just to look

Opening a saved workspace defaults to viewing only. We can inspect its list and
totals without starting work. Closing this viewer also leaves the saved list in
place.

The catalogs have no operation records yet: making a work list did not start an
attempt.

In [ ]:
with SyncWorkspace.open(workspace_directory) as viewer:
    print("Saved state:", viewer.state)
    show_counts(viewer)
    assert viewer.state == "ready"
assert source.operations() == target.operations() == ()
assert target.operation_scope(target_root) == target_scope

## 10. Record the start of an attempt

We reopen the workspace for updates and explicitly name the folder this attempt
would be allowed to change. `begin()` records the start; it does not copy files.

The target catalog now has a short operation record. The source catalog is left
alone.

In [ ]:
workspace = resources.enter_context(SyncWorkspace.open(workspace_directory, writable=True))
authority = ManagedRootAuthority(spec.operation_id, target_root, "sync")
workspace.begin(source, target, authority=authority)
print("Workspace state:", workspace.state)
record = target.operation(spec.operation_id)
assert record is not None
print("Recorded attempt:", record.status)
assert source.operations() == ()
assert len(target.operations()) == 1

## 11. Record a pretend problem

Imagine that `nested/new.txt` could not be read. We record that failure without
trying to copy the file. The empty file is still waiting, and the note remains
already present.

The work list now explains what happened instead of treating everything as a
success.

In [ ]:
with closing(workspace.batches(batch_size=2)) as batches:
    for batch in batches:
        for item in batch:
            if item.source_relative_path == PurePosixPath("nested/new.txt"):
                workspace.record_outcome(
                    item.entry_id, "failed", error_code="practice_problem",
                    error_detail="Pretend this file could not be read.",
                )
show_counts(workspace)
counts = workspace.summary()
assert (counts.skipped, counts.failed, counts.unattempted) == (1, 1, 1)

## 12. Save the finishing decision, then pause

`freeze()` saves exactly which results the attempt would finish with. It keeps
that decision fixed so a retry cannot quietly turn into a different request.
There are no successful new files in this example.

Saving the decision does **not** finish the attempt or update the target inventory.
We record an interruption and close the workspace, as if we had stopped here.

In [ ]:
saved_decision = workspace.freeze()
assert saved_decision.counts.registered == 0
assert workspace.state == "finalizing"
workspace.require_recovery()
workspace.close()
assert workspace_directory.exists()
assert target.operation_scope(target_root) == target_scope
print("The unfinished attempt has been saved for attention.")

## 13. Reopen the unfinished attempt

Archiver remembers that this attempt needs attention. An explicit resume checks
that the catalogs still match the starting point. In a real copy attempt it would
also need fresh checks of any files already placed; this practice attempt placed
none.

The saved decision stays the same. There is still no completion receipt, which
means we cannot call this a finished sync. Actual finishing belongs to 005.4.

In [ ]:
workspace = resources.enter_context(SyncWorkspace.open(workspace_directory, writable=True))
print("Reopened state:", workspace.state)
assert workspace.state == "recovery_required"
workspace.resume(source, target, authority=authority, checks=())
assert workspace.state == "finalizing"
assert workspace.frozen_request == saved_decision
assert workspace.acknowledge(target) is None
print("The saved decision is unchanged. No completion has been confirmed.")

## 14. Keep unfinished work until we choose to discard it

Ordinary cleanup refuses to remove this unfinished attempt. That leaves the list
available for inspection instead of losing the explanation of what happened.

We then deliberately discard our practice attempt. This removes its workspace
and records that the attempt was discarded. Discarding an attempt never undoes
physical copies; none were made in this lesson.

In [ ]:
try:
    workspace.dispose(target)
except OperationError:
    print("The unfinished workspace was kept for inspection.")
else:
    raise AssertionError("An unfinished attempt should not be cleaned up automatically.")
assert workspace_directory.exists()

workspace.dispose(target, discard=True)
assert not workspace_directory.exists()
record = target.operation(spec.operation_id)
assert record is not None and record.status == "discarded"
print("The practice workspace is gone. Its short operation record says:", record.status)

## 15. Check that the practice data is unchanged

The source files, target files, and both starting inventories should be unchanged.
The source catalog should also have exactly the same saved bytes. The target
catalog has added a short record saying our attempt was discarded.

These checks help show that a saved work list is different from actually changing
the files or their inventory.

In [ ]:
assert file_facts(source_root) == source_before
assert file_facts(target_root) == target_before
assert source.operation_scope(source_root) == source_scope
assert target.operation_scope(target_root) == target_scope
assert source.database_path.read_bytes() == source_database_before
assert source.operations() == ()
print("The original files and inventories are unchanged.")

## 16. Remove the practice folder

Close everything we opened and remove the temporary practice folder. If you stop
partway through the lesson, you can run this cell on its own after step 1 to clean
up. In a real operation, you would keep an unfinished workspace until you had
inspected or recovered it.

In [ ]:
resources.close()
assert not sandbox.exists()
print("The practice folder has been removed.")

## One more thing to try

Restart the kernel, change the reading batch size in step 7 to `1`, and run all
cells again. You should see three pages instead of two, while the whole-list
totals stay the same.

A common mistake is to read `ready` or `finalizing` as "the files are now synced."
They describe the saved work list. A confirmed completion and updated inventory
are separate steps.

For more examples, see the [workspace guide](../../docs/operation-workspaces.md),
the [Catalog API](../../docs/catalog-api.md#operation-inspection-plan-0053), and
[Plan 005.3](../../docs/plans/005.3-operation-records-and-execution-contract.md).